# A2A Researcher 서버

SDK 1.x의 Card·JSON-RPC·Message를 실제로 실행합니다. Researcher 답변은 교육용 문서 발췌입니다. 장기 Task·취소·스트리밍은 지원하지 않습니다. 별도 실행은 `uv run python -m courseware.a2a_server`입니다.

구현 정본은 [a2a_server.py](a2a_server.py)입니다. 아래에서 현재 파일을 읽고 import하므로 Python 수정이 자동으로 반영됩니다. 수업 본문은 [통합 교안](../../lessons/)을 사용하세요.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/courseware").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"


In [ ]:
from IPython.display import display, Code
display(Code((ROOT / "src/courseware/a2a_server.py").read_text(encoding="utf-8"), language="python"))

## 작은 실행과 검토

입력·출력·실패 조건을 예상하고 실행하세요. 원문 코드의 구현과 실제 결과를 대조합니다.

In [ ]:
import httpx
from uuid import uuid4
import a2a.types as t
from a2a.client import A2ACardResolver, ClientConfig
from a2a.client.base_client import BaseClient
from a2a.client.transports.jsonrpc import JsonRpcTransport
from courseware.a2a_server import build_a2a_app
url = "http://training.local"
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=build_a2a_app(url)), base_url=url, headers={"A2A-Version": "1.0"}) as http:
    card = await A2ACardResolver(http, url).get_agent_card()
    print(card.name)
    client = BaseClient(card, ClientConfig(streaming=False), JsonRpcTransport(http, card, url + "/"), [])
    message = t.Message(role=t.Role.ROLE_USER, message_id=str(uuid4()), parts=[t.Part(text="RAG 검색")])
    async for event in client.send_message(t.SendMessageRequest(message=message)):
        assert event.HasField("message")
        print(event.message.parts[0].text)
        assert "[D1]" in event.message.parts[0].text